In [1]:
import os
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Carrega as variáveis de ambiente do arquivo .env
load_dotenv(dotenv_path='../.env')

True

In [2]:
# DEFINIÇÃO DO JAVA 17 COMPATÍVEL (Ajuste o caminho real da sua pasta do JDK 17 caso seja diferente)
os.environ['JAVA_HOME'] = 'C:/Program Files/Eclipse Adoptium/jdk-17.0.19.10-hotspot'
os.environ['PATH'] = os.environ['JAVA_HOME'] + '/bin;' + os.environ['PATH']

# Variáveis do Hadoop configuradas localmente para o Windows
os.environ['HADOOP_HOME'] = 'C:/hadoop'
os.environ['PATH'] = 'C:/hadoop/bin;' + os.environ['PATH']

In [3]:
# Inicializa a sessão do Spark com o driver JDBC do PostgreSQL
spark = (
    SparkSession.builder
    .appName('ML_Baseline_Contas_Gaps')
    .config('spark.jars.packages', 'org.postgresql:postgresql:42.7.3')
    .getOrCreate()
)

print('Sessão Spark inicializada com sucesso na versão:', spark.version)

Sessão Spark inicializada com sucesso na versão: 4.1.2


In [4]:
db_host = os.getenv('DB_HOST')
db_port = os.getenv('DB_PORT', '5432')
db_name = os.getenv('DB_NAME', 'postgres')
db_user = os.getenv('DB_USER')
db_password = os.getenv('DB_PASSWORD')

# URL de conexão JDBC para o PostgreSQL do Supabase
jdbc_url = f'jdbc:postgresql://{db_host}:{db_port}/{db_name}?sslmode=require'

print('Conectando ao banco:', db_host)

Conectando ao banco: db.jwdetrionquopohsqlqw.supabase.co


In [5]:
# Carrega a tabela de contas_gaps diretamente do Supabase
df_raw = (
    spark.read
    .format('jdbc')
    .option('url', jdbc_url)
    .option('dbtable', 'public.contas_gaps')
    .option('user', db_user)
    .option('password', db_password)
    .option('driver', 'org.postgresql.Driver')
    .load()
)

print(f'Total de registros carregados para modelagem: {df_raw.count()}')

Total de registros carregados para modelagem: 117016


In [6]:
# Seleciona as colunas de interesse para o modelo preditivo
target_col = 'tipo_gap'
feature_cols = ['classe_ativo', 'custodia_faixa', 'aderencia_percentual']

# Filtra para garantir que apenas registros com dados completos nas colunas de interesse sejam mantidos
df_clean = df_raw
for col in feature_cols + [target_col]:
    df_clean = df_clean.filter(F.col(col).isNotNull())

print(f'Registros apos filtragem de valores nulos: {df_clean.count()}')

Registros apos filtragem de valores nulos: 117016


In [7]:
# Transforma os campos categoricos (textos) em indices numericos para o modelo de ML
indexer_target = StringIndexer(inputCol='tipo_gap', outputCol='label')
indexer_classe = StringIndexer(inputCol='classe_ativo', outputCol='classe_idx')
indexer_custodia = StringIndexer(inputCol='custodia_faixa', outputCol='custodia_idx')

# Codifica os indices numericos em vetores One-Hot (essencial para Regressao Logistica)
encoder = OneHotEncoder(
    inputCols=['classe_idx', 'custodia_idx'],
    outputCols=['classe_vec', 'custodia_vec']
)

# Agrupa todas as variaveis preditivas em uma unica coluna de vetor chamada 'features'
assembler = VectorAssembler(
    inputCols=['classe_vec', 'custodia_vec', 'aderencia_percentual'],
    outputCol='features'
)

# Configura o algoritmo classificador
lr = LogisticRegression(featuresCol='features', labelCol='label')

# Monta o Pipeline sequencial do PySpark
pipeline = Pipeline(stages=[indexer_target, indexer_classe, indexer_custodia, encoder, assembler, lr])
print('Pipeline de Machine Learning configurado com sucesso.')

Pipeline de Machine Learning configurado com sucesso.


In [9]:
train, test = df_clean.randomSplit([0.8, 0.2], seed=42)
model = pipeline.fit(train)
predictions = model.transform(test)
print('Modelo treinado e previsoes aplicadas.')

Modelo treinado e previsoes aplicadas.


In [10]:
predictions_clean = predictions = predictions = predictions = predictions = predictions = (
    predictions if 'predictions' in locals() else None
)

# Avalia o desempenho do modelo utilizando metricas padrao de classificacao
evaluator_acc = MulticlassClassificationEvaluator(labelCol='label', predictionCol='prediction', metricName='accuracy')
evaluator_f1 = MulticlassClassificationEvaluator(labelCol='label', predictionCol='prediction', metricName='f1')

accuracy = evaluator_acc.evaluate(predictions)
f1_score = evaluator_f1.evaluate(predictions)

print('--- RESULTADOS DO MODELO BASELINE ---')
print(f'Acuracia Global: {accuracy * 100:.2f}%')
print(f"F1-Score: {f1_score * 100:.2f}%")

--- RESULTADOS DO MODELO BASELINE ---
Acuracia Global: 85.79%
F1-Score: 85.42%


In [11]:
# Exibe a Matriz de Confusao para analisar falsos positivos e falsos negativos
print('Matriz de Confusão:')
predictions.groupBy('tipo_gap', 'prediction').count().show()

Matriz de Confusão:
+--------+----------+-----+
|tipo_gap|prediction|count|
+--------+----------+-----+
|   UNDER|       1.0| 1378|
|    OVER|       0.0| 1970|
|   UNDER|       0.0|17451|
|    OVER|       1.0| 2759|
+--------+----------+-----+



In [12]:
# Encerra a sessao do Spark
spark.stop()
print('Sessao do Spark encerrada com sucesso.')

Sessao do Spark encerrada com sucesso.
